# Navneet Vivek — DistilBERT sentiment robustness
Use a GPU runtime. Place the complete source project folder (without .venv or caches) at `MyDrive/Navneet_DL_Project` in your own Google Drive. This notebook runs the same source modules as Windows; it does not upload to Git or Hugging Face. Generated models and data persist in that Drive folder. Review STATUS.md before running completed stages again.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, subprocess, sys
project = Path('/content/drive/MyDrive/Navneet_DL_Project')
assert (project / 'config.json').exists(), 'Copy the project source folder into Drive first'
os.chdir(project)
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['NLTK_DATA'] = '/content/nltk_data'
def run(*args):
    subprocess.run([sys.executable, *args], check=True)


In [ ]:
run('-m', 'pip', 'install', '-r', 'requirements.txt')
run('-m', 'nltk.downloader', '-d', os.environ['NLTK_DATA'], 'wordnet', 'omw-1.4')


If Colab requests a session restart after installation, restart and rerun the mount/setup cell. Then continue below. No token or paid service is required by the code.

In [ ]:
import torch
print(torch.__version__, torch.cuda.is_available())
assert torch.cuda.is_available(), 'Choose Runtime > Change runtime type > GPU'
run('-m', 'pytest', '-q')


If your team already has split indices, pass `--split-file` to preparation before generating data. All team members must reuse those indices and the frozen perturbations.

In [ ]:
if not Path('data/prepared').exists():
    run('-m', 'navneet_dl.prepare')


In [ ]:
if not Path('models/smoke/experiment.json').exists():
    run('-m', 'navneet_dl.train', '--smoke')


In [ ]:
if not Path('models/distilbert/experiment.json').exists():
    run('-m', 'navneet_dl.train')
# To resume an interrupted training run, use:
# run('-m', 'navneet_dl.train', '--resume', 'runs/distilbert/checkpoint-N')
run('-m', 'navneet_dl.data_stats')


In [ ]:
if not Path('data/perturbed/manifest.json').exists():
    run('-m', 'navneet_dl.perturb')
run('-m', 'navneet_dl.verify')


Inspect `data/perturbed/manual_review.csv` and record semantic/sentiment validity. The generated edits are random stress tests; they are not certified meaning-preserving attacks.

In [ ]:
run('-m', 'navneet_dl.evaluate')
run('-m', 'navneet_dl.report')


In [ ]:
import pandas as pd
from IPython.display import display, Image
display(pd.read_csv('results/full/comparison.csv'))
display(Image(filename='results/full/accuracy.png'))


In [ ]:
run('-m', 'navneet_dl.predict', 'The acting was excellent and I enjoyed this film.')
